# The Format Task Replication. For Meeting 5/11/26 with Eric Xia.

# Cloning the Repository into the User Level Google Collab Storage
(Loading the repository must be ran every single time. This is because the user level memory in collab is wiped out in every new session.)

In [1]:
import os

# 1. Clear local space and move to local fast storage
%cd /content/
!rm -rf mult-ster # Remove old local clones if they exist

# 2. Clone the repo LOCALLY
!git clone -b multiplicative-steering https://github.com/sidagrw/mult-ster.git
%cd /content/mult-ster/third_party/microsoft_llm_steer_instruct

# 3. Create a local high-speed cache for Hugging Face models
# This prevents the script from re-downloading or slow-reading from Drive
os.environ['HF_HOME'] = '/content/huggingface_cache'
os.makedirs('/content/huggingface_cache', exist_ok=True)

/content
Cloning into 'mult-ster'...
remote: Enumerating objects: 99, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 99 (delta 14), reused 99 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (99/99), 3.45 MiB | 9.57 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/mult-ster/third_party/microsoft_llm_steer_instruct


# Installing the Dependencies

In [ ]:
!pip install -r requirements.txt
!pip install --quiet transformers==4.40.0 accelerate bitsandbytes transformer-lens==1.17.0 datasets hydra-core

  Using cached transformer_lens-2.15.0-py3-none-any.whl.metadata (12 kB)
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
Using cached transformer_lens-2.15.0-py3-none-any.whl (189 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.0
    Uninstalling transformers-4.40.0:
      Successfully uninstalled transformers-4.40.0
  Attempting uninstall: transformer_lens
    Found existing installation: transformer-lens 1.17.0
    Uninstalling transformer-lens-1.17.0:
      Successfully uninstalled transformer-lens-1.17.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


# Hugging Face Authentication Login
(This is necessary to access Huggingface gated local models like llama or mistral.)

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: fineGrained).
The token `EricRecreation` has been saved to /content/huggingface_cache/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-auth

In [ ]:
import transformers
# 2. Force-patch the TRANSFORMERS_CACHE bug
if not hasattr(transformers, 'TRANSFORMERS_CACHE'):
    transformers.TRANSFORMERS_CACHE = os.path.join(os.path.expanduser("~"), ".cache/huggingface/hub")

# Compute Representations

In [ ]:
import pandas as pd
import json
import os

# 1. Paths
data_path = "/content/mult-ster/third_party/microsoft_llm_steer_instruct/data/format/ifeval_augmented_filtered.jsonl"
backup_path = data_path + ".bak"

# 2. Backup the original if we haven't yet
if not os.path.exists(backup_path):
    !cp {data_path} {backup_path}
    print("Created backup of original data.")

# 3. Load the data "Properly"
print("Loading data...")
with open(backup_path, 'r') as f:
    data = [json.loads(line) for line in f]

df = pd.DataFrame(data)

# 4. Perform the Shuffle
# frac=1 means 100% of the data, random_state=42 ensures it's reproducible
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. VERIFY COLUMNS (So we don't get KeyErrors later)
print("\n✅ Data Columns Verified:")
print(df_shuffled.columns.tolist())

# 6. Save it back to the original filename in JSONL format
df_shuffled.to_json(data_path, orient='records', lines=True, force_ascii=False)

print(f"\n🚀 SUCCESS: The dataset is now properly shuffled and saved to {data_path}")
print("Even if you run a small subset now, you will have all instruction types.")

Created backup of original data.
Loading data...

✅ Data Columns Verified:
['single_instruction_kwargs', 'single_instruction_id', 'prompt_without_instruction', 'prompt', 'uid', 'key', 'instruction_id_list_original', 'kwargs', 'instruction_id_list', 'instruction_id_list_for_eval']

🚀 SUCCESS: The dataset is now properly shuffled and saved to /content/mult-ster/third_party/microsoft_llm_steer_instruct/data/format/ifeval_augmented_filtered.jsonl
Even if you run a small subset now, you will have all instruction types.


In [ ]:
import pandas as pd
import json

data_path = "/content/mult-ster/third_party/microsoft_llm_steer_instruct/data/format/ifeval_augmented_filtered.jsonl"

with open(data_path, 'r') as f:
    data = [json.loads(line) for line in f]
df = pd.DataFrame(data)

# Name Script A generates
name_a = df['single_instruction_id'].unique()[0]
# Name Script B looks for
name_b = df['instruction_id_list_for_eval'].iloc[0][0]

print(f"Script A (Compute) will save a file named: {name_a.replace(':', '_')}.h5")
print(f"Script B (Search) will look for a file named: {name_b.replace(':', '_')}.h5")

if name_a.replace(':', '_') != name_b.replace(':', '_'):
    print("\n❌ THE NAMES DO NOT MATCH. This is why the search script is crashing.")
else:
    print("\n✅ The names match. The issue is likely the folder structure (subset vs all).")

Script A (Compute) will save a file named: combination_two_responses.h5
Script B (Search) will look for a file named: combination_two_responses.h5

✅ The names match. The issue is likely the folder structure (subset vs all).


In [ ]:
%cd /content/mult-ster/third_party/microsoft_llm_steer_instruct/

!python format/compute_representations.py \
    model_name="Qwen/Qwen1.5-1.8B-Chat" \
    use_data_subset=True \
    data_subset_ratio=0.1 \
    +batch_size=128

/content/mult-ster/third_party/microsoft_llm_steer_instruct
/content/mult-ster/third_party/microsoft_llm_steer_instruct/format/compute_representations.py:21: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=config_path, config_name='compute_representations')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
device: cuda
data_path: data/format/ifeval_augmented_filtered.jsonl
dry_run: false
transformers_cache_dir: null
model_name: Qwen/Qwen1.5-1.8B-Chat
max_generation_length: 2
num_final_tokens: 1
use_data_subset: true
data_subset_ratio: 0.1
batch_size: 128

Loading model from Qwen/Qwen1.5-1.8B-Chat
/usr/local/lib/python3.12/di

# Find Best Layer

In [ ]:
!python format/find_best_layer.py \
    model_name="Qwen/Qwen1.5-1.8B-Chat" \
    representations_folder="subset_0.1" \
    n_examples_per_instruction=1 \
    max_generation_length=100 \
    +batch_size=1

/content/mult-ster/third_party/microsoft_llm_steer_instruct/format/find_best_layer.py:28: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=config_path, config_name='find_best_layer')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
device: cuda
data_path: data/format/ifeval_augmented_filtered.jsonl
output_path: layer_search_out
dry_run: false
model_name: Qwen/Qwen1.5-1.8B-Chat
max_generation_length: 100
n_examples_per_instruction: 1
include_instructions: true
cross_model_steering: false
transformers_cache_dir: null
seed: 42
steering: adjust_rs
steering_weight: 1.0
representations_folder: subset_0.1
batch_size: 1

Loading mod

# Evaluate

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

!python format/evaluate.py model_name="Qwen/Qwen1.5-1.8B-Chat"

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


/content/mult-ster/third_party/microsoft_llm_steer_instruct/format/evaluate.py:24: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path=config_path, config_name='format_evaluation')
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
device: cuda
dry_run: false
data_path: data/format/ifeval_single_instr_format.jsonl
output_path: out
transformers_cache_dir: null
model_name: Qwen/Qwen1.5-1.8B-Chat
hf_model: true
max_generation_length: 2048
include_instructions: false
cross_model_steering: false
steering: none
source_layer_idx: -1
steering_weight: 1.0
representations_folder: all
use_perplexity: true

Loading model from Qwen/Qwen1.5-1.

In [ ]:
import pandas as pd
import json

# Exact path from your screenshot
target_file = "format/out/Qwen/Qwen1.5-1.8B-Chat/no_instr/out.jsonl"

try:
    with open(target_file, 'r') as f:
        data = [json.loads(line) for line in f]

    df = pd.DataFrame(data)

    # Calculate the mean of the True/False column
    acc = df['follow_all_instructions'].mean() * 100

    print("\n" + "="*30)
    print(f"📊 REPLICATION ACCURACY")
    print(f"Model: Qwen1.5-1.8B-Chat")
    print(f"Samples: {len(df)}")
    print(f"Instruction Following: {acc:.2f}%")
    print("="*30)

except Exception as e:
    print(f"❌ Error reading file: {e}")


📊 REPLICATION ACCURACY
Model: Qwen1.5-1.8B-Chat
Samples: 163
Instruction Following: 4.29%


In [ ]:
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
!zip -r /content/drive/MyDrive/Qwen_Representations_Backup.zip /content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/

  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/Qwen/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/Qwen/Qwen1.5-1.8B-Chat/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/Qwen/Qwen1.5-1.8B-Chat/subset_0.1/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/Qwen/Qwen1.5-1.8B-Chat/subset_0.1/keywords_frequency_at least.h5 (deflated 89%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/Qwen/Qwen1.5-1.8B-Chat/subset_0.1/language_response_language_ur.h5 (deflated 75%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/representations/Qwen/Qwen1.5-1.8B-Chat/subset_0.1/detectable_content_postscript.h5 (deflated 89%

In [ ]:
!zip -r /content/drive/MyDrive/Qwen_out_Backup.zip /content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/

  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/Qwen/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/Qwen/Qwen1.5-1.8B-Chat/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/Qwen/Qwen1.5-1.8B-Chat/no_instr/ (stored 0%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/Qwen/Qwen1.5-1.8B-Chat/no_instr/args.json (deflated 36%)
  adding: content/mult-ster/third_party/microsoft_llm_steer_instruct/format/out/Qwen/Qwen1.5-1.8B-Chat/no_instr/out.jsonl (deflated 70%)
